# Introduction to the `SphereAssembly` class

In [1]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

from softmobility import SphereAssembly, Sphere

## Input with YAML format

In [2]:
yaml_data = """
dof_names:
    - length
    - angle

defaults:
    length: 1.0
    angle: 0.0

spheres:
  - # Sphere 0
    radius: 1.0
    position: [-1, 0, 0]
    orientation: [0, 0, angle]
    
  - # Sphere 1
    radius: 0.5
    position: [length, 0, 0]
    orientation: [0, 0, 0]
"""

In [3]:
# Creating SoftPlankton object
mysphereassembly = SphereAssembly(yaml_data)

  Found variables: angle, length, 
    Sphere 0
      Radius: 1.00000000000000
      Position: ['-1', '0', '0']
      Orientation: ['0', '0', 'angle']
      Coupling matrix C_H:
[[], [], [], [], [], []]
      Coupling matrix C_K:
[['0', '0'], ['0', '0'], ['0', '0'], ['0', '0'], ['0', '0'], ['0', '0']]
    Sphere 1
      Radius: 0.500000000000000
      Position: ['length', '0', '0']
      Orientation: ['0', '0', '0']
      Coupling matrix C_H:
[[], [], [], [], [], []]
      Coupling matrix C_K:
[['0', '0'], ['0', '0'], ['0', '0'], ['0', '0'], ['0', '0'], ['0', '0']]


## Input with Python lambda callable functions

In [4]:
# Creating SoftPlankton object
mysphereassembly = SphereAssembly()

# Creating degrees of freedom
mysphereassembly.add_dof(name="length", default=1.0)
mysphereassembly.add_dof(name="angle", default=0.0)
i_length = mysphereassembly.dof_variables.index("length")
i_angle  = mysphereassembly.dof_variables.index("angle")

# Adding spheres to the assembly
mysphereassembly.add_sphere(Sphere(  # Sphere 0
        position=lambda dofs, design:  np.array([-1, 0, 0]),
        orientation=lambda dofs, design: np.array([0,0,dofs[i_angle]]),
        radius=1
    ))

mysphereassembly.add_sphere(Sphere(  # Sphere 1
        position=lambda dofs, design:  np.array([dofs[i_length], 0, 0]),
        orientation=lambda dofs, design: np.array([0,0,0]),
        radius=0.5
    ))

print(repr(mysphereassembly))

NEW degrees of freedom
 ['length'] 
with default values
 [1.]
NEW degrees of freedom
 ['length', 'angle'] 
with default values
 [1. 0.]
SPHERE ASSEMBLY
  2 spheres
  2 degrees of freedom
  0 design parameters
  0 input parameters

Default values
  degrees of freedom dof: ['length', 'angle'] = [1. 0.]
  design parameters param: [] = []
  input parameters param: []

SPHERE 0
  radius: 1.0
  position: [-1  0  0]
  orientation: [0. 0. 0.]
  c_field:
[]
  c_stiff:
[]

SPHERE 1
  radius: 0.5
  position: [1. 0. 0.]
  orientation: [0 0 0]
  c_field:
[]
  c_stiff:
[]



## Visualization of the sphere assembly

### Useful plotting utility functions

In [5]:
def rotate_points(points, axis_angle):
    """
    Rotate an array of points using Rodrigues' rotation formula.
    """
    points = np.array(points)
    theta = np.linalg.norm(np.array(axis_angle))
    if theta < 1e-8:  # No rotation
        return points
    
    k = axis_angle / theta  # Unit rotation axis
    kx, ky, kz = k
    
    K = np.array([
        [0, -kz, ky],
        [kz, 0, -kx],
        [-ky, kx, 0]
    ])
    
    R = np.eye(3) + np.sin(theta) * K + (1 - np.cos(theta)) * np.dot(K, K)  # Rodrigues formula
    
    return np.dot(points, R.T)

In [6]:
def create_sphere(radius=1, center=(0, 0, 0), orientation=(0, 0, 0), resolution=32, checkered_scale=8):
    """
    Create a 3D checkered sphere with sharp transitions.
    """
    u = np.linspace(0, 2 * np.pi, resolution)
    v = np.linspace(0, np.pi, resolution)

    x = radius * np.outer(np.cos(u), np.sin(v))
    y = radius * np.outer(np.sin(u), np.sin(v))
    z = radius * np.outer(np.ones(np.size(u)), np.cos(v))

    # Stack points and apply rotation
    points = np.column_stack([x.ravel(), y.ravel(), z.ravel()])
    rotated_points = rotate_points(points, orientation)

    x_rot = rotated_points[:, 0].reshape(x.shape) + center[0]
    y_rot = rotated_points[:, 1].reshape(y.shape) + center[1]
    z_rot = rotated_points[:, 2].reshape(z.shape) + center[2]

    # Generate sharp checkered pattern by controlling color block size
    checkered_u = (np.floor(np.linspace(0, resolution, resolution) / checkered_scale) % 2).astype(int)
    checkered_v = (np.floor(np.linspace(0, resolution, resolution) / checkered_scale) % 2).astype(int)
    colors = np.add.outer(checkered_u, checkered_v) % 2  # 0s and 1s for high contrast

    return x_rot, y_rot, z_rot, colors

In [7]:
def add_arrow(fig, start, direction, color, name):
    """
    Adds a 3D arrow (line + cone) to the figure.
    """
    end = start + direction
    cone_start = start + 0.75 * direction
    arrow_length = np.linalg.norm(direction)

    # Line (shaft)
    fig.add_trace(go.Scatter3d(
        x=[start[0], end[0]], y=[start[1], end[1]], z=[start[2], end[2]], 
        mode="lines", line=dict(color=color, width=5), name=f"{name}-axis"
    ))

    # Cone (arrowhead)
    fig.add_trace(go.Cone(
        x=[cone_start[0]], y=[cone_start[1]], z=[cone_start[2]],
        u=[direction[0]], v=[direction[1]], w=[direction[2]], 
        colorscale=[[0, color], [1, color]], showscale=False,
        sizemode="absolute", sizeref=0.333 * arrow_length  # Adjust arrowhead size
    ))

In [8]:
def update_figure(fig, assembly, dofs=None, params=None):
    """
    Efficiently update the existing 3D visualization of SphereAssembly.
    """
    dofs_array = np.array(dofs) if dofs is not None else assembly.dof_defaults
    params_array = np.array(params) if params is not None else assembly.param_defaults

    # Clear previous data while keeping the figure object
    fig.data = []

    # Add 3D coordinate arrows
    add_arrow(fig, start=np.array([0, 0, 0]), direction=np.array([1, 0, 0]), color='blue', name="X")
    add_arrow(fig, start=np.array([0, 0, 0]), direction=np.array([0, 1, 0]), color='red', name="Y")
    add_arrow(fig, start=np.array([0, 0, 0]), direction=np.array([0, 0, 1]), color='green', name="Z")

    # Plot spheres with transparency
    for i_sphere, sphere in enumerate(assembly.spheres):
        x, y, z, colors = create_sphere(
            radius=sphere.radius(dofs_array, params_array),
            center=sphere.position(dofs_array, params_array),
            orientation=sphere.orientation(dofs_array, params_array)
        )
        fig.add_trace(go.Surface(
            x=x, y=y, z=z, surfacecolor=colors,
            showscale=False, 
            cmin=0, cmax=assembly.Nspheres / (1 + i_sphere),  
            colorscale='Thermal',  # Change colormap to 'Cividis'
        ))

    # Improve layout: bigger figure, remove axes, center spheres
    fig.update_layout(
        title="3D Sphere Assembly",
        autosize=False,  # Disable autosizing
        width=900,  # Set width in pixels
        height=900,  # Set height in pixels  
        scene=dict(     
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
        #     camera=dict(eye=dict(x=1.5, y=0.5, z=0.5))  # Better viewing angle
        ),
        scene_camera=dict(projection=dict(type="orthographic"))  # <-- Isometric projection
    )

    fig.show()    

### Plot with sliders

In [9]:
# Creating sliders
length_slider = widgets.FloatSlider(min=0, max=4, step=0.2, value=2, description='Length')
angle_slider = widgets.FloatSlider(min=-4, max=4, step=0.5, value=0, description='Angle')

# Create initial figure
fig = go.Figure()

# Wrapper function to extract slider values
def update_wrapper(length, angle):
    dofs = np.array([length, angle])
    update_figure(fig, mysphereassembly, dofs, np.array([]))

# Use interactive widget
interactive_plot = widgets.interactive(update_wrapper, length=length_slider, angle=angle_slider)

# Display the interactive widget
display(interactive_plot)

interactive(children=(FloatSlider(value=2.0, description='Length', max=4.0, step=0.2), FloatSlider(value=0.0, …